# Tools  🔨
Los tools le permiten a los agents 'actuar' en el mundo real.
Descripciones cuidadosas pueden ayudar a tu agent a descubrir cómo usar tus tools.

LangChain soporta muchos formatos y conjuntos de tools. Aquí cubriremos algunos casos comunes, pero consulta la [documentación](https://docs.langchain.com/oss/python/langchain/tools) para más información.

## Configuración

Cargar y/o verificar las variables de entorno necesarias

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

required_vars = ["GEMINI_API_KEY"]

for var in required_vars:
    value = os.environ.get(var)
    if value:
        print(f"✅ {var} = {value[:8]}...{value[-4:]}")
    else:
        print(f"❌ {var} no está definida")

## Ejemplo: Calculadora

En este ejemplo, el docstring y los argumentos inferidos con sus tipos son usados por el LLM para determinar cuándo y cómo llamar al tool.

In [ ]:
from typing import Literal

from langchain.tools import tool


@tool
def real_number_calculator(
    a: float, b: float, operation: Literal["add", "subtract", "multiply", "divide"]
) -> float:
    """Perform basic arithmetic operations on two real numbers."""
    print("Invocando tool de calculadora")
    # Ejecutar la operación especificada
    if operation == "add":
        return a + b
    elif operation == "subtract":
        return a - b
    elif operation == "multiply":
        return a * b
    elif operation == "divide":
        if b == 0:
            raise ValueError("Division by zero is not allowed.")
        return a / b
    else:
        raise ValueError(f"Invalid operation: {operation}")

In [ ]:
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI

agent = create_agent(
    model=ChatGoogleGenerativeAI(model="gemini-flash-latest"),
    tools=[real_number_calculator],
    system_prompt="Eres un asistente útil",
)

Esto invoca tu tool de calculadora.

In [ ]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Cuanto es 3.1125 * 4.1234"}]}
)
print(result["messages"][-1].content)

Podemos revisar el **metadata** en [LangSmith Observability](https://smith.langchain.com/public/b77bde6c-f0ad-4256-bfab-7d514ece3405/r) para ver esto.

La descripción del tool puede tener un gran impacto.
Es posible que esto **no** invoque tu tool de calculadora porque los inputs son enteros. (los resultados varían de ejecución a ejecución)

In [ ]:
result = agent.invoke({"messages": [{"role": "user", "content": "cuanto es 3 * 4"}]})
print(result["messages"][-1].content)

Esto frecuentemente tampoco invoca el tool, aunque los inputs sean números reales. (los resultados varían de ejecución a ejecución)

In [ ]:
result = agent.invoke({"messages": [{"role": "user", "content": "cuanto es 3.0 * 4.0"}]})
print(result["messages"][-1].content)

## Agregar una descripción más detallada
Aunque una descripción básica suele ser suficiente, LangChain tiene soporte para descripciones mejoradas. El ejemplo a continuación usa un método: descripciones de argumentos estilo Google (Google Style). Al usar `parse_docstring=True`, se parsean y pasan las descripciones de los args al modelo. Puedes renombrar el tool y cambiar su descripción, lo cual es útil cuando compartes un tool estándar pero quieres instrucciones específicas para un agent.

In [ ]:
from typing import Literal

from langchain.tools import tool


@tool(
    "calculator",
    parse_docstring=True,
    description=(
        "Realiza operaciones aritméticas básicas con dos números reales. "
        "Úsalo siempre que haya operaciones con números, incluso si son enteros."
    ),
)
def real_number_calculator(
    a: float, b: float, operation: Literal["add", "subtract", "multiply", "divide"]
) -> float:
    """Realiza operaciones aritméticas básicas con dos números reales.

    Args:
        a (float): El primer número.
        b (float): El segundo número.
        operation (Literal["add", "subtract", "multiply", "divide"]):
            La operación aritmética a realizar.

            - `"add"`: Devuelve la suma de `a` y `b`.
            - `"subtract"`: Devuelve el resultado de `a - b`.
            - `"multiply"`: Devuelve el producto de `a` y `b`.
            - `"divide"`: Devuelve el resultado de `a / b`. Lanza un error si `b` es cero.

    Returns:
        float: El resultado numérico de la operación especificada.

    Raises:
        ValueError: Si se proporciona una operación inválida o se intenta dividir por cero.
    """
    print("- Invocando tool de calculadora")
    if operation == "add":
        return a + b
    elif operation == "subtract":
        return a - b
    elif operation == "multiply":
        return a * b
    elif operation == "divide":
        if b == 0:
            raise ValueError("No se permite la división por cero.")
        return a / b
    else:
        raise ValueError(f"Operación inválida: {operation}")


In [ ]:
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI

agent = create_agent(
    model=ChatGoogleGenerativeAI(model="gemini-flash-latest"),
    tools=[real_number_calculator],
    system_prompt="Eres un asistente útil",
)

In [ ]:
result = agent.invoke({"messages": [{"role": "user", "content": "cuanto es 4 por 4"}]})
print(result["messages"][-1].content)

In [ ]:
for i, msg in enumerate(result["messages"]):
    msg.pretty_print()

Revisemos el [trace de LangSmith Observability](https://smith.langchain.com/public/7d65902c-bd3c-4fc6-bbd3-7c1d66566fda/r) para ver la descripción del tool.

In [ ]:
result = agent.invoke({"messages": [{"role": "user", "content": "what is 3 * 4"}]})
print(result["messages"][-1].content)

## Actividad
Crea tu propio tool y pruébalo aquí.

In [ ]:
from typing import Literal

@tool(
    "convertidor_temperaturas",
    parse_docstring=True,
    description=(
        "Convierte temperaturas entre Celsius, Fahrenheit y Kelvin. "
        "Úsalo siempre que el usuario pregunte por conversiones de temperatura."
    ),
)
def your_tool(
    valor: float,
    de: Literal["celsius", "fahrenheit", "kelvin"],
    a: Literal["celsius", "fahrenheit", "kelvin"],
) -> float:
    """Convierte una temperatura entre escalas.

    Args:
        valor (float): Valor de temperatura a convertir.
        de (Literal["celsius", "fahrenheit", "kelvin"]): Escala de origen.
        a (Literal["celsius", "fahrenheit", "kelvin"]): Escala de destino.

    Returns:
        float: Temperatura convertida a la escala indicada.
    """
    print(f"- Convirtiendo {valor}° {de} → {a}")
    if de == "celsius":
        celsius = valor
    elif de == "fahrenheit":
        celsius = (valor - 32) * 5 / 9
    else:
        celsius = valor - 273.15

    if a == "celsius":
        return round(celsius, 2)
    elif a == "fahrenheit":
        return round(celsius * 9 / 5 + 32, 2)
    else:
        return round(celsius + 273.15, 2)

In [ ]:
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI

agent = create_agent(
    model=ChatGoogleGenerativeAI(model="gemini-flash-latest"),
    tools=[your_tool],
    system_prompt="Eres un asistente científico. Usa la herramienta de conversión de temperaturas para dar resultados exactos.",
)

result = agent.invoke({"messages": [{"role": "user", "content": "¿Cuánto es 100°C en Fahrenheit y en Kelvin?"}]})
print(result["messages"][-1].content)